In [1]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
from decimal import Decimal,ROUND_FLOOR

%matplotlib inline
%matplotlib notebook

## 1.Load file.

In [2]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [3]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

## 2. Groundwater ###

In [4]:
iters = np.shape(P_atm)[0] # total timestep.

In [5]:
gw_measure = 0 # we do not consider 'measure' for the time being.

### 2.1 Assumptions

__1.__ The infiltration water from the open paved area flows directly to groundwater (percolation).

__2.__ The groundwater area is equal to the total area, minus the surface water area and part of the paved roof area of which the basement is not above groundwater;

__3.__ Drainage and seepage are based on the groundwater level at the end of the previous time step. Drainage and seepage are somewhat reduced due to the changing groundwater level caused by the fluxes;



### 2.2 Build up using default settings in excel.

In original excel file, the data in the location \$BP$9 is set as 0 instead of 114.4836, in order to remove the measure.

#### 2.2.1 Input data preparation

__a.__ percolation from unsaturated zone (p_uz_gw) and from open paved (p_op_gw), taken directly from excel as input.

In [6]:
p_uz_gw =[0,0,0.005657707,0.005618768,0.005607055,0.51359524,1.570722358,0.504984486,0.780915564,-0.042925624,-0.038992226,-0.039023253
,0.214837853,0.230212491,0.230129816,-0.040361628,-0.039069533,-0.003253121,0.005523032,0.0054696,0.0054583,0.005446826,0.00543538
,0.005423959,0.005412565,0.005401197,0.005389854,0.005378538,0.005367247,0.005355983,0.00335004,-0.024577547,-0.024456084,-0.024468189
,-0.024479632,-0.024491052,-0.024502446,-0.024513814,-0.024525156,1.318861553,0.243597104,-0.001896437,0.005258825,0.283566848,0.003867561
,0.005201287,0.005184018,0.288074732,0.003805768,0.2880624,1.701196382,1.997073739,1.995675416,1.995690433,1.980978135,1.774968174,1.775954262
,1.775969813,1.775976046,1.775982312]
p_op_gw = [0,0,0,0,0,0,0.041666667,0.041666667,0.041666667,0,0,0,0.041666667,0.041666667,0.041666667,0,0,0,0,0,0,0,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0.041666667,0.041666667,0,0,0.041666667,0,0,0,0.041666667,0,0.041666667,0.041666667,0.041666667,0.041666667,0.041666667,0.041666667
,0.041666667,0,0,0,0]

__b.__ open water level, taken from excel.

In [7]:
owl = np.ones(iters) * 1.5

#### 2.2.2 Using Arrays to build up the structure first.

In [8]:
soilmatrix = pd.read_csv(path + 'soilparameter_new.csv')
etmatrix = pd.read_csv(path + 'ETparameter.csv')

In [9]:
def ETSelector(a, b):
    #a soil type, b crop type
    sol = etmatrix.loc[(etmatrix.soil_type == int(a)) & (etmatrix.crop_type == int(b))]
    return sol

In [10]:
# Selector is modified a bit from the "SoilSelector&ETSelector" to avoid multiple repeated reference of selector to improve efficiency.
def SoilSelector(a, b, c):
    #a soil type, b crop type, c GWL [m -MSL]
    if c>= 0.0 and c <= 2.5:
        c = float(Decimal(str(c)).quantize(Decimal('.1'), rounding=ROUND_FLOOR))
    elif c < 3.0:
        c = 2.5
    elif c < 5.0:
        c = int(c)
    elif c <= 10:
        c = 5.0
    else:
        c = 10.0
    rootzone_thickness = 100 * ETSelector(a, b)['th_rz_m'].values
    
    sol = soilmatrix.loc[(soilmatrix.soil_type == int(a)) & (soilmatrix.th_rz == int(rootzone_thickness)) & (soilmatrix.gwl == c)]
    return sol

In [11]:
# areas for different area
pr_area = 1560
pr_part_aboveGW = 0.0
cp_area = 803.3906406
op_area = 481.6093594
up_area = 6855
uz_area = up_area
ow_area = 300
ow_part_aboveGW = 0.0

gw_area = pr_area * pr_part_aboveGW + cp_area + op_area + up_area + ow_area * ow_part_aboveGW

In [12]:
delta_t = 1 / 24

seep_def = 0
w = 100 # drainage resistance
vc = 20000 # vertical hydraulic resistance
h_deepgw = 21.5 #
flux = 1
init_gwl = 1.5

gwl = np.zeros(iters)
gwl[0] = init_gwl
gwl_sl = np.zeros(iters)
sum_p_gw = np.zeros(iters)
r_meas_gw = np.zeros(iters)
sc_gw = np.zeros(iters)
sc_gw[0] = SoilSelector(2, 1, init_gwl)['stor_coef'].values
gwl_up = np.zeros(iters)
gwl_low = np.zeros(iters)
h_gw = np.zeros(iters)
s_out = np.zeros(iters)
d_ow = np.zeros(iters)

t = 1 

while t <= iters - 1:
    
    # Total percolation from unsaturated zone and from open paved area to groundwater    
    sum_p_gw[t] = (p_uz_gw[t] * uz_area + p_op_gw[t] * op_area) / gw_area 
    
    # Inflow from measure area (if applicable), set as 0 for the time being
    r_meas_gw[t] = 0
    
    # because sc part uses gwl_up and gwl_low in unsaturated part, so gwl_up and gwl_low is calculated here again. When
    # all the individual modules are merged together, we could be able remove this repeated calculation. 
    # gwl_up
    c = gwl[t-1]
    if c>= 0.0 and c <= 2.5:
        c = float(Decimal(str(c)).quantize(Decimal('.1'), rounding=ROUND_FLOOR))
    elif c < 3.0:
        c = 2.5
    elif c < 5.0:
        c = int(c)
    elif c <= 10:
        c = 5.0
    else:
        c = 10.0
    gwl_up[t] = c
    
    # gwl_low
    if gwl_up[t] < 2.5:
        gwl_low[t] = gwl_up[t] + 0.1
    elif gwl_up[t] < 3:
        gwl_low[t] = 3
    elif gwl_up[t] < 4:
        gwl_low[t] = 4
    elif gwl_up[t] < 5:
        gwl_low[t] = 5
    else:
        gwl_low[t] = 10
        
    # Storage coefficient of the groundwater for the current time step
    if gwl[t-1] < 10:
        sc_gw[t] = SoilSelector(2, 1, gwl_low[t])['stor_coef'].values +(gwl_low[t] - gwl[t-1]) / (gwl_low[t] - gwl_up[t]) * (SoilSelector(2, 1, gwl_up[t])['stor_coef'].values-SoilSelector(2, 1, gwl_low[t])['stor_coef'].values)
    else:
        sc_gw[t] = SoilSelector(2, 1, 10)['stor_coef'].values
    
    # Groundwater level at the end of the current time step [m-SL].
    if seep_def > 0.5:
        h_gw[t] = -(((sum_p_gw[t] + r_meas_gw[t]) / 1000 * w * vc - h_deepgw * w - owl[t-1] * vc) / (w + vc) + (-(gwl[t-1] + gwl_sl[t-1]) - ((sum_p_gw[t] + r_meas_gw[t]) / 1000 * w * vc - h_deepgw * w - owl[t -1] * vc) / (w + vc)) * np.exp(- delta_t * (w + vc) /(sc_gw[t] * w * vc)))
    else:
        h_gw[t] = - (w * (((sum_p_gw[t] + r_meas_gw[t]) - flux) / 1000) - owl[t-1] + (-(gwl[t-1] + gwl_sl[t-1]) - (w * (((sum_p_gw[t] + r_meas_gw[t])- flux) / 1000) - owl[t-1])) * np.exp(- delta_t / (sc_gw[t] * w )))
    
    # Downward seepage flux to deep groundwater during current time step.
    if seep_def < 0.5:
        s_out[t] = delta_t * flux
    else:
        s_out[t] = 1000 * (h_deepgw - 0.5 * (h_gw[t] + (gwl[t-1] + gwl_sl[t-1]))) / c * delta_t
        
    # Groundwater drainage to the open water for the current time step [mm].
    d_ow[t] = sum_p_gw[t] + r_meas_gw[t] - s_out[t] - sc_gw[t] * (gwl[t-1] + gwl_sl[t-1] -  h_gw[t]) * 1000        
 
    
    # groundwater level below surface level at the end of the current time step [m-SL].
    gwl[t] = max(0, gwl[t-1] - (sum_p_gw[t] + r_meas_gw[t] - s_out[t] - d_ow[t]) / (1000 * sc_gw[t]))
    
    # groundwater level above surface level at the end of the current time step [m-SL]
    gwl_sl[t] = -1 * max(0, (0 - (gwl[t-1] - (sum_p_gw[t] + r_meas_gw[t] - s_out[t] - d_ow[t])/(1000 * sc_gw[t]))) * sc_gw[t]) 
    t += 1
    
filename = 'Results_Groundwater_arraybuildup_test.csv'
np.savetxt('sol/' + filename, np.c_[sum_p_gw,r_meas_gw,sc_gw,h_gw,s_out, d_ow, gwl,gwl_sl], fmt = "%.8f", delimiter=',', header = 'sum_p_gw,r_meas_gw,sc_gw,h_gw,s_out, d_ow, gwl,gwl_sl') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated with excel.')

The results have been validated with excel.


#### 2.2.3 Using Class to build up the module

In [13]:
class Groundwater:
    def __init__(self, gw_area, seep_def = 0, w = 100, vc = 20000, h_deepgw = 21.5, flux = 1, init_gwl = 1.5, croptype = 2, soiltype = 1):
        
        # state
        self.init_gwl = init_gwl
        
        # parameter
        self.gw_area =  gw_area
        self.seep_def = seep_def
        self.w =  w
        self.vc = vc
        self.h_deepgw = h_deepgw
        self.flux = flux
        self.soiltype = soiltype
        self.croptype = croptype
        self.prev_gwl = init_gwl
        self.prev_gwl_sl = 0
    
    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are current precipitation and evaporation.'
    
    def sol(self, p_uz_gw, uz_area, p_op_gw, prev_owl, op_area, delta_t = 1 / 24): 
        
        # sum_p_gw
        sum_p_gw = (p_uz_gw * uz_area + p_op_gw * op_area) / self.gw_area

        # Inflow from measure area (if applicable), set as 0 for the time being
        r_meas_gw = 0
        
        # gwl_up
        c = float(self.prev_gwl)
        if c>= 0.0 and c <= 2.5:
            c = float(Decimal(str(c)).quantize(Decimal('.1'), rounding=ROUND_FLOOR))
        elif c < 3.0:
            c = 2.5
        elif c < 5.0:
            c = int(c)
        elif c <= 10:
            c = 5.0
        else:
            c = 10.0
        gwl_up = c
    
        # gwl_low
        if gwl_up < 2.5:
            gwl_low = round(gwl_up + 0.1, 2)
        elif gwl_up < 3:
            gwl_low = 3
        elif gwl_up < 4:
            gwl_low = 4
        elif gwl_up < 5:
            gwl_low = 5
        else:
            gwl_low = 10
        
        # Storage coefficient of the groundwater for the current time step
        if self.prev_gwl < 10:
            sc_gw = SoilSelector(self.soiltype, self.croptype, gwl_low)['stor_coef'].values + (gwl_low - self.prev_gwl) / (gwl_low - gwl_up) * (SoilSelector(self.soiltype, self.croptype, gwl_up)['stor_coef'].values - SoilSelector(self.soiltype, self.croptype, gwl_low)['stor_coef'].values)
        else:
            sc_gw = SoilSelector(self.soiltype, self.croptype, 10)['stor_coef'].values
        
        # Groundwater level at the end of the current time step [m-SL].
        if self.seep_def > 0.5:
            h_gw = -(((sum_p_gw + r_meas_gw) / 1000 * self.w * self.vc - self.h_deepgw * self.w - prev_owl * self.vc) / (self.w + self.vc) + (-(self.prev_gwl + self.prev_gwl_sl) - ((sum_p_gw + r_meas_gw) / 1000 * self.w * self.vc - self.h_deepgw * self.w - prev_owl * self.vc) / (self.w + self.vc)) * np.exp(- delta_t * (self.w + self.vc) /(sc_gw * self.w * self.vc)))
        else:
            h_gw = - (self.w * (((sum_p_gw + r_meas_gw) - self.flux) / 1000) - prev_owl + (-(self.prev_gwl + self.prev_gwl_sl) - (self.w * (((sum_p_gw + r_meas_gw)- self.flux) / 1000) - prev_owl)) * np.exp(- delta_t / (sc_gw * self.w )))         
       
        # Downward seepage flux to deep groundwater during current time step.
        if self.seep_def < 0.5:
            s_out = delta_t * self.flux
        else:
            s_out = 1000 * (self.h_deepgw - 0.5 * (h_gw + (self.prev_gwl + self.prev_gwl_sl))) / self.vc * delta_t
        
        # Groundwater drainage to the open water for the current time step [mm].
        d_ow = sum_p_gw + r_meas_gw - s_out - sc_gw * (self.prev_gwl + self.prev_gwl_sl -  h_gw) * 1000        
 
        # Groundwater level below surface level at the end of the current time step [m-SL].
        gwl = max(0, self.prev_gwl - (sum_p_gw + r_meas_gw - s_out - d_ow) / (1000 * sc_gw))
    
        # Groundwater level above surface level at the end of the current time step [m-SL]
        gwl_sl = -1 * max(0, (0 - (self.prev_gwl - (sum_p_gw + r_meas_gw - s_out - d_ow)/(1000 * sc_gw))) * sc_gw)          
        
            
        # update state
        self.prev_gwl = gwl
        self.prev_gwl_sl = gwl_sl
         
        return sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl

In [14]:
t = 1

sum_p_gw = [0] 
r_meas_gw = [0]
gwl_up = [0]
gwl_low = [0]
sc_gw = [SoilSelector(2, 1, 1.5)['stor_coef'].values]
h_gw = [0]
s_out = [0]
d_ow = [0]
gwl = [1.5]
gwl_sl= [0]

# Specify the parameter or use the default setting.
m = Groundwater(gw_area, seep_def = 0, w = 100, vc = 20000, h_deepgw = 21.5, flux = 1, init_gwl = 1.5, croptype = 2, soiltype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.ow_area = 300
    sol = m.sol(p_uz_gw[t], 6855, p_op_gw[t], prev_owl = owl[t-1], op_area = 481.6093594, delta_t = 1 / 24)
    
    sum_p_gw.append(sol[0])
    r_meas_gw.append(sol[1])
    gwl_up.append(sol[2]) 
    gwl_low.append(sol[3]) 
    sc_gw.append(sol[4])
    h_gw.append(sol[5])
    s_out.append(sol[6])
    d_ow.append(sol[7])
    gwl.append(sol[8])
    gwl_sl.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_Groundwater_c1s1.csv'
np.savetxt('sol/' + filename, np.c_[sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl], fmt = "%.8f", delimiter=',', header = 'sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

The results have been validated.


### 2.3 Validate with excel with two sets of coefficients.

#### 2.3.1 C2S1

Drainage resistance 80; 
seepage = 0;
flux = 2;
init_gwl = 1.6;
h_deepgw = 23;
flow resistence  = 30000;

In [15]:
p_uz_gw =[0,0,0.003901544,0.003874952,0.003865634,0.511856264,1.569180044,0.503849016,0.779374474,-0.04435691,-0.040736896,-0.040762633,0.213100847
,0.228575657,0.228500871,-0.041988157,-0.040797471,-0.004977735,0.003814369,0.003766582,0.003757626,0.003748523,0.003739442,0.003730384,0.003721349
,0.003712335,0.003703344,0.003694376,0.003685429,0.003676505,0.001672899,-0.026253103,-0.026139764,-0.026149457,-0.026158588,-0.026167698
,-0.026176786,-0.026185851,-0.026194894,1.31719411,0.242432631,-0.003459818,0.003607269,0.281920633,0.002327956,0.003558838,0.003544802
,0.28643778,0.002276781,0.286429203,1.699671954,1.996072664,1.994784866,1.994802864,1.980094524,1.774083058,1.774996868,1.775015654,1.775025501
,1.775035363]
p_op_gw = [0,0,0,0,0,0,0.041666667,0.041666667,0.041666667,0,0,0,0.041666667,0.041666667,0.041666667,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0.041666667,0.041666667,0,0,0.041666667,0,0,0,0.041666667,0,0.041666667,0.041666667,0.041666667,0.041666667,0.041666667,0.041666667
,0.041666667,0,0,0,0]

__Note that__ when you change the __draiange resistance, flux and init_gwl__, the gwl is automatically changed, __so the input percolation from unsaturated zone is also changed.

In [16]:
t = 1

sum_p_gw = [0] 
r_meas_gw = [0]
gwl_up = [0]
gwl_low = [0]
sc_gw = [SoilSelector(2, 1, 1.6)['stor_coef'].values]
h_gw = [0]
s_out = [0]
d_ow = [0]
gwl = [1.6]
gwl_sl= [0]


# Specify the parameter or use the default setting.
m = Groundwater(gw_area, seep_def = 0, w = 80, vc = 30000, h_deepgw = 23, flux = 2, init_gwl = 1.6, croptype = 2, soiltype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.ow_area = 300
    sol = m.sol(p_uz_gw[t], 6855, p_op_gw[t], prev_owl = owl[t-1], op_area = 481.6093594, delta_t = 1 / 24)
    
    sum_p_gw.append(sol[0])
    r_meas_gw.append(sol[1])
    gwl_up.append(sol[2]) 
    gwl_low.append(sol[3]) 
    sc_gw.append(sol[4])
    h_gw.append(sol[5])
    s_out.append(sol[6])
    d_ow.append(sol[7])
    gwl.append(sol[8])
    gwl_sl.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_Groundwater_c2s1.csv'
np.savetxt('sol/' + filename, np.c_[sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl], fmt = "%.8f", delimiter=',', header = 'sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

The results have been validated.


#### 2.3.2 C2S2

Drainage resistance 120; 
seepage = 1;
flux = 1.5;
init_gwl = 1.3;
h_deepgw = 20;
flow resistence  = 25000;
Soiltype = 3 
croptype = 1

In [17]:
p_uz_gw =[0,0,0.012454488,0.01235691,0.012314248,0.520271547,1.577586236,0.512277858,0.787725029,-0.036021096,-0.032474091,-0.032532207
,0.221298711,0.236757085,0.236652687,-0.03386665,-0.032724943,0.0030632,0.011826368,0.011748062,0.01170814,0.011668236,0.011628513
,0.011588971,0.011549608,0.011510423,0.011471415,0.011432582,0.011393924,0.011355439,0.009322423,-0.018633047,-0.01855179,-0.018590603
,-0.01862873,-0.018666688,-0.018704478,-0.0187421,-0.018779555,1.32458118,0.249958166,0.003909225,0.010918914,0.289205683,0.009622427
,0.010789775,0.010748858,0.293614775,0.009466674,0.293553116,1.706810486,1.8875,1.8875,1.8875,1.8875,1.8875,1.8875,1.8875,1.8875
,1.805383626]
p_op_gw = [0,0,0,0,0,0,0.041666667,0.041666667,0.041666667,0,0,0,0.041666667,0.041666667,0.041666667,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0.041666667,0.041666667,0,0,0.041666667,0,0,0,0.041666667,0,0.041666667,0.041666667,0.041666667,0.041666667,0.041666667,0.041666667
,0.041666667,0,0,0,0]

In [18]:
t = 1

sum_p_gw = [0] 
r_meas_gw = [0]
gwl_up = [0]
gwl_low = [0]
sc_gw = [SoilSelector(3, 1, 1.3)['stor_coef'].values]
h_gw = [0]
s_out = [0]
d_ow = [0]
gwl = [1.3]
gwl_sl= [0]


# Specify the parameter or use the default setting.
m = Groundwater(gw_area, seep_def = 1, w = 120, vc = 25000, h_deepgw = 20, flux = 1.5, init_gwl = 1.3, soiltype = 3, croptype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.ow_area = 300
    sol = m.sol(p_uz_gw[t], 6855, p_op_gw[t], prev_owl = owl[t-1], op_area = 481.6093594, delta_t = 1 / 24)
    
    sum_p_gw.append(sol[0])
    r_meas_gw.append(sol[1])
    gwl_up.append(sol[2]) 
    gwl_low.append(sol[3]) 
    sc_gw.append(sol[4])
    h_gw.append(sol[5])
    s_out.append(sol[6])
    d_ow.append(sol[7])
    gwl.append(sol[8])
    gwl_sl.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_Groundwater_c2s2.csv'
np.savetxt('sol/' + filename, np.c_[sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl], fmt = "%.8f", delimiter=',', header = 'sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

The results have been validated.
